# prep.tableDivBySite

The table generated here contains the data of site diversity for virus, bacteria, plants, and hosts, and the number of cooccurrences per site. 

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from daforfer import DaforferDB
import matplotlib.pyplot as plt
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

┌──────────────────────┬───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│         name         │                                                        description                                                        │
│       varchar        │                                                          varchar                                                          │
├──────────────────────┼───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│ tablePABs            │ This table summarizes most of the information of our detected OTUs, including host_range, site_range, habitat_range, etc. │
│ tableSiteInformation │ Library sites and context                                                                                                 │
└──────────────────────┴──────────────────────────────────────────────────────────────────────────────────

## PAB diversity

In [22]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')
bacteria_hits

,site,library,habitat,n_extracts,host_taxon,taxid,scientific_name,is_pab,pab_type
0,C1,PV534,Crop,3,Diplotaxis erucoides,NaN,NaN,NaN,NaN
1,C1,PV535,Crop,17,Brassica oleracea,1563157.0,Pseudomonas endophytica,True,pab_unknown
2,C1,PV535,Crop,17,Brassica oleracea,1270.0,Micrococcus luteus,True,pab_unknown
3,C1,PV538,Crop,8,Brassica oleracea,NaN,NaN,NaN,NaN
4,C1,PV540,Crop,1,Picris echioides,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
626,Z2,PV527,Crop,4,Convolvulus arvensis,1770058.0,Devosia elaeis,True,pab_unknown
627,Z2,PV529,Crop,1,Picris echioides,47880.0,Pseudomonas fulva,True,pab_unknown
628,Z2,PV529,Crop,1,Picris echioides,1220495.0,Pseudomonas punonensis,True,pab_unknown
629,Z2,PV529,Crop,1,Picris echioides,289370.0,Pseudomonas argentinensis,True,pab_unknown


We simply create a list of item counts in one of the columns. 

In [23]:
pab_alpha_diversity = bacteria_hits.value_counts(
    ['site', 'habitat', 'scientific_name']
    ).reset_index().groupby(
        ['site', 'habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
pab_alpha_diversity

,site,habitat,hits
0,C1,Crop,"[1, 1, 1, 1, 1, 1]"
1,C2,Crop,"[2, 1, 1, 1, 1, 1, 1, 1]"
2,E1,Wasteland,"[2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
3,E2,Wasteland,"[3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
4,E3,Wasteland,"[5, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
5,E4,Wasteland,"[8, 4, 4, 4, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, ..."
6,H1,Crop,[1]
7,H2,Crop,[1]
8,H3,Crop,"[1, 1, 1, 1]"
9,L1,Edge,"[3, 3, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [24]:
pab_alpha_diversity['species_richness'] = pab_alpha_diversity['hits'].apply(lambda x: len(x))
pab_alpha_diversity = pab_alpha_diversity.sort_values(by=['habitat', 'site'])
pab_alpha_diversity['disturbed'] = pab_alpha_diversity['habitat'].apply(
    lambda x: {"Crop": "disturbed", "Wasteland": "non-disturbed", "Edge": "disturbed", "Oak": "non-disturbed"}[x]
)
pab_alpha_diversity = pab_alpha_diversity.drop(columns=['hits'])[['site', 'habitat', 'disturbed', 'species_richness']]
pab_alpha_diversity

,site,habitat,disturbed,species_richness
0,C1,Crop,disturbed,6
1,C2,Crop,disturbed,8
6,H1,Crop,disturbed,1
7,H2,Crop,disturbed,1
8,H3,Crop,disturbed,4
13,M1,Crop,disturbed,2
14,M2,Crop,disturbed,1
15,M3,Crop,disturbed,2
16,M4,Crop,disturbed,1
21,Z1,Crop,disturbed,11


## Virus diversity

In [25]:
# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')
virus_alpha_diversity = virus_hits.value_counts(
    ['site', 'habitat', 'scientific_name']
    ).reset_index().groupby(
        ['site', 'habitat']
    )['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
virus_alpha_diversity['species_richness'] = virus_alpha_diversity['hits'].apply(lambda x: len(x))
virus_alpha_diversity = virus_alpha_diversity.drop(columns=['hits'])[['site', 'habitat',  'species_richness']]
virus_alpha_diversity


,site,habitat,species_richness
0,C1,Crop,18
1,C2,Crop,12
2,E1,Wasteland,14
3,E2,Wasteland,30
4,E3,Wasteland,17
5,E4,Wasteland,28
6,H1,Crop,13
7,H2,Crop,12
8,H3,Crop,33
9,L1,Edge,54


## Plant diversity

In [26]:

plant_hits = pd.read_csv("input/plant-data-3.csv", sep=';')
plant_hits = pd.merge(
    plant_hits, metadata[['site', 'habitat']].drop_duplicates('site'), on='site', how='inner'
)
plant_alpha_diversity = plant_hits.drop_duplicates(
    ['site', 'species name'], keep='first'
).value_counts('site').reset_index().sort_values('site').rename(columns={'count': 'species_richness'})
plant_alpha_diversity

,site,species_richness
18,C1,12
12,C2,17
2,E1,54
0,E2,61
1,E3,59
5,E4,50
22,H1,9
21,H2,9
17,H3,12
8,L1,45


In [27]:
plant_hits.drop_duplicates(
    ['site', 'species name'], keep='first'
)

,species name,collection,site,habitat
0,Amaranthus_sp,C1F,C1,Crop
1,Brassica_oleracea,C1F,C1,Crop
2,Daucus_carota,C1F,C1,Crop
3,Diplotaxis_erucoides,C1F,C1,Crop
4,Malva_sylvestris,C1F,C1,Crop
...,...,...,...,...
1170,Datura_stramonium,Z4V,Z2,Crop
1171,Salsola_kali,Z4V,Z2,Crop
1172,Scabiosa_sicula,Z4V,Z2,Crop
1173,Sorghum_halepense,Z4V,Z2,Crop


The following table must be checked against some of the previous analysis. In principle, we should have 63 species in crop, 103 in edge, 117 in oak and 124 in wasteland

In [28]:
plant_hits.drop_duplicates(
    ['habitat', 'species name'], keep='first'
).value_counts('habitat').reset_index().sort_values('habitat')

,habitat,count
3,Crop,63
2,Edge,103
1,Oak,117
0,Wasteland,124


## Host diversity

In [29]:
host_alpha_diversity = metadata.value_counts(
    ['site', 'habitat', 'host_taxon']
).reset_index().groupby(
    ['site', 'habitat']
)['count'].apply(list).reset_index().rename(columns={'count': 'hits'})
host_alpha_diversity['species_richness'] = host_alpha_diversity['hits'].apply(lambda x: len(x))
host_alpha_diversity = host_alpha_diversity.drop(columns=['hits'])[['site', 'habitat',  'species_richness']]
host_alpha_diversity

,site,habitat,species_richness
0,C1,Crop,4
1,C2,Crop,5
2,E1,Wasteland,14
3,E2,Wasteland,18
4,E3,Wasteland,13
5,E4,Wasteland,18
6,H1,Crop,3
7,H2,Crop,3
8,H3,Crop,4
9,L1,Edge,28


## Merge

In [30]:
# bacteria_alpha_diversity = db.conn.query('SELECT * FROM D_PABAlphaDiv').df()
alpha_diversity = pd.merge(
    pab_alpha_diversity[['site', 'habitat', 'disturbed', 'species_richness']], 
    virus_alpha_diversity[['site', 'species_richness']],
    on='site', suffixes=['_bact', '_vir']
)

alpha_diversity = pd.merge(
    alpha_diversity,
    plant_alpha_diversity[['site', 'species_richness']].rename(
        columns={
            'species_richness': 'species_richness_plant',
        }
    ),
    on='site'
)

alpha_diversity = pd.merge(
    alpha_diversity,
    host_alpha_diversity[['site', 'species_richness']].rename(
        columns={
            'species_richness': 'species_richness_host',
        }
    ),
    on='site'
)


# alpha_diversity.to_csv("output/diversity.all.csv", sep=";", index=False)
# db.save_dataframe(
#     df=alpha_diversity, table_name="D_ADAllOrganismsSite",
#     description="Alpha diversity per site for virus, plant, host and bacteria at site level"
# )

alpha_diversity

,site,habitat,disturbed,species_richness_bact,species_richness_vir,species_richness_plant,species_richness_host
0,C1,Crop,disturbed,6,18,12,4
1,C2,Crop,disturbed,8,12,17,5
2,H1,Crop,disturbed,1,13,9,3
3,H2,Crop,disturbed,1,12,9,3
4,H3,Crop,disturbed,4,33,12,4
5,M1,Crop,disturbed,2,20,14,5
6,M2,Crop,disturbed,1,13,9,4
7,M3,Crop,disturbed,2,9,11,4
8,M4,Crop,disturbed,1,9,16,3
9,Z1,Crop,disturbed,11,14,13,6


## Cooccurrences by site

There are a couple ways of counting those.

### Number of cooccurrence cocodetections

In [31]:
cooccurrence_codetections_by_library = db.conn.sql('SELECT * FROM D_coocDetections').df()
cooccurrence_codetections_by_site = pd.merge(metadata, cooccurrence_codetections_by_library, on='library').groupby('site', as_index=False)['number_cooccurrences_per_library'].sum()
cooccurrence_codetections_by_site = cooccurrence_codetections_by_site.rename(columns={'number_cooccurrences_per_library': 'coccurrence_codetections'}) # type: ignore
cooccurrence_codetections_by_site

,site,coccurrence_codetections
0,C1,5.0
1,C2,1.0
2,E1,2.0
3,E2,2.0
4,E3,1.0
5,E4,27.0
6,H1,0.0
7,H2,1.0
8,H3,4.0
9,L1,39.0


### Number of cooccurrence detections at site level

In [32]:
cooccurrence_detection_pairs_by_library = db.conn.sql('SELECT * FROM D_coocPairDetections').df()
cooccurrence_detection_pairs_by_library

,library,pair
0,PV003bgi,Bradyrhizobium elkanii-Watermelon mosaic virus
1,PV004bgi,Bradyrhizobium elkanii-Watermelon mosaic virus
2,PV005bgi,Bradyrhizobium elkanii-Watermelon mosaic virus
3,PV007bgi,Bradyrhizobium elkanii-Watermelon mosaic virus
4,PV012bgi,Bradyrhizobium elkanii-Watermelon mosaic virus
...,...,...
344,PV544,Beet western yellows virus-Rhodococcoides fasc...
345,PV544,Rhodococcoides fascians-Rubus chlorotic mottle...
346,PV544,Agrobacterium tumefaciens-Beet western yellows...
347,PV544,Beet western yellows virus-Sphingomonas sp. Le...


In [33]:
pd.merge(metadata, cooccurrence_detection_pairs_by_library, on='library', how='left')

,site,library,habitat,n_extracts,host_taxon,pair
0,C1,PV534,Crop,3,Diplotaxis erucoides,NaN
1,C1,PV535,Crop,17,Brassica oleracea,NaN
2,C1,PV538,Crop,8,Brassica oleracea,NaN
3,C1,PV540,Crop,1,Picris echioides,NaN
4,C1,PV544,Crop,4,Sisymbrium runcinatum,Beet chlorosis virus-Rhodococcoides fascians
...,...,...,...,...,...,...
605,Z2,PV527,Crop,4,Convolvulus arvensis,Pantoea ananatis-Pepper mild mottle virus
606,Z2,PV527,Crop,4,Convolvulus arvensis,Pepper mild mottle virus-Pseudomonas oryzihabi...
607,Z2,PV527,Crop,4,Convolvulus arvensis,Pseudomonas oryzihabitans-Rubus chlorotic mott...
608,Z2,PV529,Crop,1,Picris echioides,Pseudomonas punonensis-Tobacco mild green mosa...


In [34]:
cooccurrence_detection_pairs_by_library = db.conn.sql('SELECT * FROM D_coocPairDetections').df()


cooccurrence_detections_by_site = pd.merge(metadata, cooccurrence_detection_pairs_by_library, on='library', how='inner').drop_duplicates(
    ['pair', 'site']
)[['site', 'pair']].value_counts(['site']).reset_index().rename(columns={'count': 'cooccurrence_detections'})
cooccurrence_detections_by_site

,site,cooccurrence_detections
0,L3,47
1,L1,25
2,L4,25
3,E4,14
4,Z1,9
5,L2,7
6,Z2,7
7,Q1,6
8,C1,5
9,H3,4


In [35]:
tableDivBySite = pd.merge(alpha_diversity, cooccurrence_codetections_by_site, on=['site'])
tableDivBySite = pd.merge(tableDivBySite, cooccurrence_detections_by_site, on=['site'], how='left').fillna(0)
tableDivBySite

,site,habitat,disturbed,species_richness_bact,species_richness_vir,species_richness_plant,species_richness_host,coccurrence_codetections,cooccurrence_detections
0,C1,Crop,disturbed,6,18,12,4,5.0,5.0
1,C2,Crop,disturbed,8,12,17,5,1.0,1.0
2,H1,Crop,disturbed,1,13,9,3,0.0,0.0
3,H2,Crop,disturbed,1,12,9,3,1.0,1.0
4,H3,Crop,disturbed,4,33,12,4,4.0,4.0
5,M1,Crop,disturbed,2,20,14,5,3.0,1.0
6,M2,Crop,disturbed,1,13,9,4,2.0,1.0
7,M3,Crop,disturbed,2,9,11,4,0.0,0.0
8,M4,Crop,disturbed,1,9,16,3,1.0,1.0
9,Z1,Crop,disturbed,11,14,13,6,10.0,9.0


## Save

In [36]:
db.save_dataframe(
    tableDivBySite, "D_Site_level_div",
    description="Site-level diversity and number of cooccurring virus-bacteria"
)

Saved D_Site_level_div to db.2026-02-24


In [37]:
si.save_dataframe(
    tableDivBySite, "tableDivBySite",
    description="Site-level diversity and number of cooccurring virus-bacteria"
)

Saved tableDivBySite to si.2026-02-24


In [38]:
tableDivBySite.to_csv("output/table.DiversityBySite.csv", sep=";")

In [39]:
si.conn.close()
db.conn.close()

In [40]:
tableDivBySite

,site,habitat,disturbed,species_richness_bact,species_richness_vir,species_richness_plant,species_richness_host,coccurrence_codetections,cooccurrence_detections
0,C1,Crop,disturbed,6,18,12,4,5.0,5.0
1,C2,Crop,disturbed,8,12,17,5,1.0,1.0
2,H1,Crop,disturbed,1,13,9,3,0.0,0.0
3,H2,Crop,disturbed,1,12,9,3,1.0,1.0
4,H3,Crop,disturbed,4,33,12,4,4.0,4.0
5,M1,Crop,disturbed,2,20,14,5,3.0,1.0
6,M2,Crop,disturbed,1,13,9,4,2.0,1.0
7,M3,Crop,disturbed,2,9,11,4,0.0,0.0
8,M4,Crop,disturbed,1,9,16,3,1.0,1.0
9,Z1,Crop,disturbed,11,14,13,6,10.0,9.0
